# Correlation Analysis Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Correlation Analysis**.  
It demonstrates the major types of correlation used in statistics, machine learning, engineering analytics, and time-series analysis.

Main topics covered:

1. Pearson correlation  
2. Spearman rank correlation  
3. Kendall tau  
4. Partial correlation  
5. Auto-correlation  
6. Cross-correlation  
7. Correlation matrix  
8. Small exercises and summary tables  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

## Create Example Data

We create several example datasets with linear, monotonic, confounded, and time-dependent relationships.

In [ ]:
n = 120

# Linear relationship for Pearson
x_linear = np.linspace(0, 20, n)
y_linear = 3 * x_linear + np.random.normal(0, 8, n)

# Monotonic but nonlinear relationship for Spearman/Kendall
x_mono = np.linspace(1, 10, n)
y_mono = x_mono**2 + np.random.normal(0, 5, n)

# Confounded variables for partial correlation
z_conf = np.random.normal(50, 10, n)
x_conf = 0.7 * z_conf + np.random.normal(0, 5, n)
y_conf = 0.6 * z_conf + np.random.normal(0, 5, n)

# Time series for auto-correlation and cross-correlation
t = np.arange(0, 150)
series_a = np.sin(t / 8) + 0.2 * np.random.normal(size=len(t))
series_b = np.roll(series_a, 4) + 0.2 * np.random.normal(size=len(t))

data = pd.DataFrame({
    'x_linear': x_linear,
    'y_linear': y_linear,
    'x_mono': x_mono,
    'y_mono': y_mono,
    'z_conf': z_conf,
    'x_conf': x_conf,
    'y_conf': y_conf
})

data.head()

## 1. Pearson Correlation

The **Pearson correlation coefficient** measures the strength of a **linear relationship**.

$$
r_{xy} = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum (x_i-\bar{x})^2 \sum (y_i-\bar{y})^2}}
$$

In [ ]:
pearson_r, pearson_p = stats.pearsonr(x_linear, y_linear)
pd.DataFrame({'Measure': ['Pearson r', 'p Value'], 'Value': [pearson_r, pearson_p]})

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(x_linear, y_linear)
plt.title('Pearson Example: Linear Relationship')
plt.xlabel('x_linear')
plt.ylabel('y_linear')
plt.show()

## 2. Spearman Rank Correlation

The **Spearman correlation** measures the strength of a **monotonic** relationship based on ranks.

For data without ties:

$$
\rho = 1 - \frac{6\sum d_i^2}{n(n^2-1)}
$$

In [ ]:
spearman_rho, spearman_p = stats.spearmanr(x_mono, y_mono)
pd.DataFrame({'Measure': ['Spearman rho', 'p Value'], 'Value': [spearman_rho, spearman_p]})

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(x_mono, y_mono)
plt.title('Spearman Example: Monotonic Nonlinear Relationship')
plt.xlabel('x_mono')
plt.ylabel('y_mono')
plt.show()

## 3. Kendall Tau

The **Kendall tau** coefficient measures ordinal association based on concordant and discordant pairs.

$$
\tau = \frac{C-D}{\frac{n(n-1)}{2}}
$$

In [ ]:
kendall_tau, kendall_p = stats.kendalltau(x_mono, y_mono)
pd.DataFrame({'Measure': ['Kendall tau', 'p Value'], 'Value': [kendall_tau, kendall_p]})

### Compare Pearson, Spearman, and Kendall on the Same Monotonic Data

In [ ]:
pearson_mono, _ = stats.pearsonr(x_mono, y_mono)
comparison = pd.DataFrame({
    'Correlation Type': ['Pearson', 'Spearman', 'Kendall'],
    'Coefficient': [pearson_mono, spearman_rho, kendall_tau]
})
comparison

## 4. Partial Correlation

Partial correlation measures the relationship between two variables after controlling for a third variable.

For first-order partial correlation controlling \(Z\):

$$
r_{xy\cdot z} = \frac{r_{xy}-r_{xz}r_{yz}}{\sqrt{(1-r_{xz}^2)(1-r_{yz}^2)}}
$$

In [ ]:
r_xy = stats.pearsonr(x_conf, y_conf)[0]
r_xz = stats.pearsonr(x_conf, z_conf)[0]
r_yz = stats.pearsonr(y_conf, z_conf)[0]

partial_r = (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz**2) * (1 - r_yz**2))

pd.DataFrame({
    'Measure': ['Pearson r(x,y)', 'Pearson r(x,z)', 'Pearson r(y,z)', 'Partial r(x,y|z)'],
    'Value': [r_xy, r_xz, r_yz, partial_r]
})

### Interpretation

If the raw Pearson correlation between `x_conf` and `y_conf` is high, but the partial correlation becomes much smaller after controlling for `z_conf`, then the apparent relationship was largely driven by the confounder.

## 5. Auto-Correlation

Auto-correlation measures how a variable is correlated with its own lagged values.

At lag \(k\):

$$
\rho_k = \frac{\sum_{t=k+1}^{n}(x_t-\bar{x})(x_{t-k}-\bar{x})}{\sum_{t=1}^{n}(x_t-\bar{x})^2}
$$

In [ ]:
def autocorr(series, lag):
    series = np.asarray(series)
    return np.corrcoef(series[lag:], series[:-lag])[0, 1]

lags = range(1, 16)
acf_values = [autocorr(series_a, lag) for lag in lags]

pd.DataFrame({'Lag': list(lags), 'Auto-correlation': acf_values}).head(10)

In [ ]:
plt.figure(figsize=(8, 4))
plt.stem(list(lags), acf_values)
plt.title('Auto-correlation of series_a')
plt.xlabel('Lag')
plt.ylabel('Auto-correlation')
plt.show()

## 6. Cross-Correlation

Cross-correlation measures similarity between two series across lags.

A normalized lagged form is:

$$
\rho_{xy}(k)=\frac{\sum_t (x_t-\bar{x})(y_{t-k}-\bar{y})}{\sqrt{\sum_t (x_t-\bar{x})^2 \sum_t (y_t-\bar{y})^2}}
$$

In [ ]:
def crosscorr(x, y, lag):
    x = np.asarray(x)
    y = np.asarray(y)
    if lag > 0:
        return np.corrcoef(x[lag:], y[:-lag])[0, 1]
    elif lag < 0:
        lag = abs(lag)
        return np.corrcoef(x[:-lag], y[lag:])[0, 1]
    else:
        return np.corrcoef(x, y)[0, 1]

lag_values = range(-12, 13)
ccf_values = [crosscorr(series_a, series_b, lag) for lag in lag_values]

pd.DataFrame({'Lag': list(lag_values), 'Cross-correlation': ccf_values}).head(10)

In [ ]:
plt.figure(figsize=(9, 4))
plt.stem(list(lag_values), ccf_values)
plt.title('Cross-correlation between series_a and series_b')
plt.xlabel('Lag')
plt.ylabel('Cross-correlation')
plt.show()

### Interpret the Peak Lag

The lag with the highest cross-correlation often indicates the delay between the two series.

In [ ]:
best_lag = list(lag_values)[int(np.argmax(ccf_values))]
best_corr = max(ccf_values)
best_lag, best_corr

## 7. Correlation Matrix

A correlation matrix summarizes pairwise correlations among several variables.

For multiple variables:

$$
R = [r_{ij}]_{p \times p}
$$

In [ ]:
corr_matrix = data[['x_linear', 'y_linear', 'x_mono', 'y_mono', 'z_conf', 'x_conf', 'y_conf']].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(corr_matrix, interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title('Correlation Matrix')
plt.show()

## 8. Small Summary Table

This table gathers the main correlation results from the notebook.

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        'Pearson r(x_linear, y_linear)',
        'Spearman rho(x_mono, y_mono)',
        'Kendall tau(x_mono, y_mono)',
        'Pearson r(x_conf, y_conf)',
        'Partial r(x_conf, y_conf | z_conf)',
        'Best auto-correlation at lag 1',
        'Best cross-correlation'
    ],
    'Value': [
        pearson_r,
        spearman_rho,
        kendall_tau,
        r_xy,
        partial_r,
        acf_values[0],
        best_corr
    ]
})
summary

## 9. Mini Exercises

Try these on your own:

1. Add an extreme outlier to the Pearson example and see how the coefficient changes.  
2. Compare Pearson and Spearman on a strongly nonlinear but monotonic dataset.  
3. Build another confounded example and compute the partial correlation.  
4. Change the lag used to generate `series_b` and see how the cross-correlation peak shifts.  
5. Replace the synthetic data with your own engineering dataset and compute a correlation matrix.  
6. Investigate whether optimizer performance ranks are better compared with Spearman or Kendall.

These exercises are especially useful for AI, machine learning, structural engineering, optimization, and time-series analysis.